# Multiscale 1D RM-CLEAN

Standard (Hogbom) RM-CLEAN models the Faraday dispersion function (FDF) as a sum
of delta functions. For a **Faraday-thick** source, whose FDF is genuinely
extended in Faraday depth, that becomes a picket fence of deltas: it still fits
the data, but the model is a spray of components that also soaks up noise.
Multiscale RM-CLEAN (Cornwell 2008; Offringa & Smirnov 2017) cleans with
extended kernels, so an extended feature is modelled by a few broad components
instead.

**Multiscale is a wideband tool.** Whether it can do anything is decided by the
coverage, not by a knob. Two scales set the window:

- **Resolution**: the Faraday thickness $\Delta\phi$ must exceed the RMSF FWHM,
  else the source is a point as far as CLEAN can tell.
- **Recoverability**: $\Delta\phi$ must be below the largest recoverable scale
  $\phi_{\rm max} = \pi / \lambda^2_{\rm min}$ (set by the highest frequency),
  else the structure is resolved out and depolarised, and no algorithm recovers
  it.

The auto scale grid encodes this. When $\phi_{\rm max}$ leaves no room for an
extended kernel, the grid collapses to `[0.0]` and multiscale silently reduces to
single-scale (and says so). We walk three regimes:

- **A, degenerate**: a single narrow band (RACS-low). Grid `[0.0]`, multiscale
  equals single-scale.
- **B, engaging**: wide, contiguous coverage (GMIMS 0.3-1.8 GHz). Extended grid;
  multiscale models the thick flux that survives as a few broad components.
- **C, forced**: the explicit `multiscale_scales` escape hatch on narrow
  coverage, and what it does (and does not) buy.

**On counting.** These runs have no major cycle (no invert-and-subtract against
the data), so there are two counts. Single-scale places one delta per **minor
iteration** (`n_iter`). Multiscale's `n_iter` is the number of **minor cycles**
(scale re-selections), each running an inner per-scale Hogbom loop; its total
component-placement count is `n_sub_minor_iter`. Multiscale reaches convergence
in far fewer *minor cycles*, but the honest work comparison is
`n_sub_minor_iter` against single-scale's `n_iter`, and by that measure
multiscale places **more** components in these (realistic, sidelobe-rich)
RMSFs, not fewer. So we quote both and do not sell multiscale as "faster": its
value is the model, not the component count.

## Setup

In [ ]:
from __future__ import annotations

import logging
import logging.handlers

import astropy.units as u
import matplotlib.pyplot as plt
import numpy as np
from numpy.typing import NDArray
from rm_lite.tools_1d import rmclean, rmsynth
from rm_lite.utils.clean import MultiscaleOptions, default_scales
from rm_lite.utils.logging import quiet_logs
from rm_lite.utils.synthesis import freq_to_lambda2

plt.rcParams["figure.dpi"] = 110
rng = np.random.default_rng(7)

# Source and CLEAN constants, shared by every case.
RM_RADM2 = 25.0  # true Faraday depth of the source
PSI0_DEG = 10.0  # intrinsic polarisation angle
FRAC_POL = 0.5  # intrinsic (peak) fractional polarisation
RMS_NOISE = 0.02  # noise per Q, U channel
AUTO_MASK = 10.0  # mask threshold, in units of the FDF noise
AUTO_THRESHOLD = 0.5  # clean threshold in FDF noise units; the mask lets us go deep
MULTISCALE_MAX_ITER_SUB_MINOR = 3000
SCALE_BIAS = 0.6  # library default (WSClean's); keeps a thin source on the delta scale

DELTA_THIN = 0.0
DELTA_MARGINAL = 1.0
DELTA_RESOLVED_GMIMS = 6.0


def burn_slab(
    lambda_sq_arr_m2: NDArray[np.float64], delta_rm_radm2: float
) -> NDArray[np.complex128]:
    """Burn slab P(lambda^2); delta_rm=0 is a Faraday-thin source."""
    return (
        FRAC_POL
        * np.exp(2j * (np.deg2rad(PSI0_DEG) + RM_RADM2 * lambda_sq_arr_m2))
        * np.sinc(delta_rm_radm2 * lambda_sq_arr_m2 / np.pi)
    ).astype(np.complex128)


def two_thin_points(
    lambda_sq_arr_m2: NDArray[np.float64], separation_radm2: float
) -> NDArray[np.complex128]:
    """Two Faraday-thin components at RM +/- separation/2, same total flux."""
    return (
        FRAC_POL
        * np.exp(2j * (np.deg2rad(PSI0_DEG) + RM_RADM2 * lambda_sq_arr_m2))
        * np.cos(separation_radm2 * lambda_sq_arr_m2)
    ).astype(np.complex128)


def auto_grid(synth) -> NDArray[np.float64]:
    """The auto scale grid (RMSF FWHM units) run_rmclean_from_synth would pick."""
    phi = synth.fdf_arrs["phi_arr_radm2"].to_numpy().astype(float)
    fwhm = float(synth.fdf_parameters["fwhm_rmsf_radm2"][0])
    phi_max = float(synth.fdf_parameters["phi_max_scale_radm2"][0])
    return default_scales(phi, fwhm, MultiscaleOptions(), phi_max_scale_radm2=phi_max)


def model_span(res) -> int:
    """Channels carrying model flux: wide for extended scales, ~1 for a delta."""
    model = np.abs(res.fdf_arrs["fdf_model_complex_arr"].to_numpy().astype(complex))
    return int((model > 0.01 * model.max()).sum())


def clean_stats(res) -> tuple[float, int, int]:
    """(mom0, minor iterations, component steps) for a 1D clean result.

    Single-scale places one component per minor iteration, so the two counts are
    equal. Multiscale's minor count is scale re-selections; `n_sub_minor_iter` is
    the total per-scale Hogbom steps, i.e. the count directly comparable to a
    single-scale iteration count.
    """
    mom0 = float(res.fdf_parameters["mom0_debias"][0])
    minor = int(np.ravel(res.clean_parameters["n_iter"])[0])
    comps = int(np.ravel(res.clean_parameters["n_sub_minor_iter"])[0])
    return mom0, minor, comps

One plot function, a 2x2 grid comparing single-scale (left) and multiscale
(right). The top row shows the FDF (true, dirty, clean); the bottom row shows the
CLEAN model and residual with the mask and threshold levels. The true FDF is a
delta for a thin source, a tophat for a slab, two deltas for the two-point case.</cell id="3">

In [ ]:
def plot_clean(synth, single, multi, source_kind, extent, rmsf_fwhm, title):
    """2x2 comparison. Columns: single-scale vs multiscale. Top row: the FDF
    (true, dirty, clean). Bottom row: the CLEAN model, residual, and the mask and
    threshold levels. Multiscale's minor-cycle and sub-minor totals are annotated
    separately."""
    phi = synth.fdf_arrs["phi_arr_radm2"].to_numpy().astype(float)
    dirty = synth.fdf_arrs["fdf_dirty_complex_arr"].to_numpy().astype(complex)

    # True (intrinsic) Faraday dispersion function: delta / tophat / two deltas.
    true_model = np.zeros_like(phi)
    if source_kind == "thin":
        true_model[np.argmin(np.abs(phi - RM_RADM2))] = FRAC_POL
    elif source_kind == "slab":
        true_model[np.abs(phi - RM_RADM2) <= extent / 2] = FRAC_POL
    elif source_kind == "points":
        for centre in (RM_RADM2 - extent / 2, RM_RADM2 + extent / 2):
            true_model[np.argmin(np.abs(phi - centre))] = FRAC_POL / 2

    half = max(4 * rmsf_fwhm, 0.8 * extent)
    fig, axs = plt.subplots(2, 2, sharex=True, sharey="row", figsize=(10, 6))
    for col, (res, name) in enumerate(
        zip((single, multi), ("single-scale", "multiscale"), strict=True)
    ):
        clean = res.fdf_arrs["fdf_clean_complex_arr"].to_numpy().astype(complex)
        model = res.fdf_arrs["fdf_model_complex_arr"].to_numpy().astype(complex)
        resid = res.fdf_arrs["fdf_residual_complex_arr"].to_numpy().astype(complex)
        mask = float(res.clean_parameters["mask"][0])
        threshold = float(res.clean_parameters["threshold"][0])
        _, minor, comps = clean_stats(res)
        count = (
            f"{minor} iters."
            if name == "single-scale"
            else f"{minor} iters.,\n{comps} sub-minor"
        )

        # Top row: the FDF.
        top = axs[0, col]
        top.plot(phi, true_model, color="0.5", ls="--", lw=1.2, label="true FDF")
        top.plot(phi, np.abs(dirty), color="k", alpha=0.35, label="dirty")
        top.plot(phi, np.abs(clean), color="k", label="clean")
        top.step(phi, np.abs(model), color="tab:red", where="mid", label="model")
        top.plot(phi, np.abs(resid), color="tab:blue", alpha=0.6, label="residual")
        top.axhline(mask, color="tab:orange", ls=":", lw=0.8, label="mask")
        top.axhline(threshold, color="tab:green", ls=":", lw=0.8, label="threshold")
        top.text(
            0.02,
            0.95,
            f"{name}: {count}",
            transform=top.transAxes,
            va="top",
            fontweight="bold",
        )
        top.set(ylabel="|FDF|")

        # Bottom row: model, residual, mask and threshold.
        bot = axs[1, col]
        bot.step(phi, np.abs(model), color="tab:red", where="mid", label="model")
        bot.plot(phi, np.abs(resid), color="tab:blue", alpha=0.6, label="residual")
        bot.axhline(mask, color="tab:orange", ls=":", lw=0.8, label="mask")
        bot.axhline(threshold, color="tab:green", ls=":", lw=0.8, label="threshold")
        bot.set(
            ylabel="|model|, |residual|",
            xlabel=rf"$\phi$ / {u.rad / u.m**2:latex_inline}",
            xlim=(RM_RADM2 - half, RM_RADM2 + half),
        )
    axs[0, 0].legend(fontsize=7, loc="upper right")
    axs[1, 0].legend(fontsize=7, loc="upper right")
    fig.suptitle(title)
    fig.tight_layout()

## Regime A: degenerate (narrowband)

RACS-low is a single 288 MHz band (744-1032 MHz). Its largest recoverable
Faraday scale $\phi_{\rm max} = \pi / \lambda^2_{\rm min}$ is *smaller* than the
RMSF FWHM, so no extended scale kernel fits the $\phi$ window. The auto grid
collapses to `[0.0]` and multiscale is single-scale; `default_scales` raises a
warning to say so.

This is physically correct, not a failure: Faraday structure wider than
$\phi_{\rm max}$ is depolarised beyond recovery for such narrow coverage, so
there is no extended flux to model in the first place.

In [ ]:
# RACS-low: a single 288 MHz band.
freq_hz = np.linspace(744e6, 1032e6, 144)
lambda_sq = freq_to_lambda2(freq_hz)

Build the RMSF, capture the degeneration warning as the auto grid is chosen, and
confirm multiscale equals single-scale on a Faraday-thin source.

In [ ]:
pol = (
    burn_slab(lambda_sq, DELTA_THIN)
    + rng.normal(0, RMS_NOISE, freq_hz.size)
    + 1j * rng.normal(0, RMS_NOISE, freq_hz.size)
).astype(np.complex128)
err = np.ones_like(pol) * (RMS_NOISE + 1j * RMS_NOISE)
with quiet_logs(logging.ERROR):
    synth = rmsynth.run_rmsynth(freq_hz, pol, err, n_samples=20, do_fit_rmsf=True)

rmsf_fwhm = float(synth.fdf_parameters["fwhm_rmsf_radm2"][0])
phi_max = float(synth.fdf_parameters["phi_max_scale_radm2"][0])

# Capture the warning default_scales raises while building the degenerate grid.
buffer = logging.handlers.BufferingHandler(1000)
logging.getLogger("rmtools-lite").addHandler(buffer)
grid = auto_grid(synth)
logging.getLogger("rmtools-lite").removeHandler(buffer)
warnings_seen = [r.getMessage() for r in buffer.buffer]

print(f"RACS-low RMSF FWHM = {rmsf_fwhm:.1f} rad/m^2")
print(
    f"max recoverable scale = {phi_max:.0f} rad/m^2 ({phi_max / rmsf_fwhm:.1f} x FWHM)"
)
print(f"auto scale grid (FWHM units) = {list(grid)}")
for msg in warnings_seen:
    print("WARNING:", msg)

# The grid degenerates to a single delta scale, and default_scales says so.
assert np.allclose(grid, [0.0])
assert any("degenerated to [0.0]" in m for m in warnings_seen)

# So multiscale here IS single-scale. Run both on the thin source and compare.
with quiet_logs(logging.ERROR):
    single = rmclean.run_rmclean_from_synth(
        synth, auto_mask=AUTO_MASK, auto_threshold=AUTO_THRESHOLD
    )
    multi = rmclean.run_rmclean_from_synth(
        synth,
        auto_mask=AUTO_MASK,
        auto_threshold=AUTO_THRESHOLD,
        multiscale=True,
        multiscale_scale_bias=SCALE_BIAS,
        multiscale_max_iter_sub_minor=MULTISCALE_MAX_ITER_SUB_MINOR,
    )

single_mom0, single_n, _ = clean_stats(single)
multi_mom0, multi_minor, multi_comps = clean_stats(multi)
print(f"single: mom0={single_mom0:.3f} iterations={single_n}")
print(
    f"multi:  mom0={multi_mom0:.3f} minor_cycles={multi_minor} "
    f"component_steps={multi_comps}"
)

# Degenerate grid: multiscale reduces to single-scale (flux matches to <1%).
assert abs(multi_mom0 - single_mom0) < 0.01
assert 0.40 < multi_mom0 < 0.60

In [ ]:
plot_clean(
    synth, single, multi, "thin", 0.0, rmsf_fwhm, "Regime A: RACS-low (degenerate grid)"
)

## Regime B: engaging (wideband)

Wide, contiguous coverage is where multiscale earns its keep. GMIMS-DRAGONS-style
coverage runs continuously from 0.3 to 1.8 GHz: the dense, wide $\lambda^2$ span
gives a fine RMSF (good Faraday resolution) and a largest recoverable scale of
roughly 19 RMSF FWHM. The auto grid is now *extended* (`[0.0, 4.0, 8.0]`), so
genuinely resolved thick structure both exists and survives, and multiscale can
model it with a few broad components rather than a picket fence of deltas.

In [ ]:
freq_hz = np.linspace(0.3e9, 1.8e9, 500)
lambda_sq = freq_to_lambda2(freq_hz)

### Thin source ($\Delta\phi = 0$)

In [ ]:
pol = (
    burn_slab(lambda_sq, DELTA_THIN)
    + rng.normal(0, RMS_NOISE, freq_hz.size)
    + 1j * rng.normal(0, RMS_NOISE, freq_hz.size)
).astype(np.complex128)
err = np.ones_like(pol) * (RMS_NOISE + 1j * RMS_NOISE)
with quiet_logs(logging.ERROR):
    synth = rmsynth.run_rmsynth(freq_hz, pol, err, n_samples=20, do_fit_rmsf=True)
    single = rmclean.run_rmclean_from_synth(
        synth, auto_mask=AUTO_MASK, auto_threshold=AUTO_THRESHOLD
    )
    multi = rmclean.run_rmclean_from_synth(
        synth,
        auto_mask=AUTO_MASK,
        auto_threshold=AUTO_THRESHOLD,
        multiscale=True,
        multiscale_scale_bias=SCALE_BIAS,
        multiscale_max_iter_sub_minor=MULTISCALE_MAX_ITER_SUB_MINOR,
    )

single_mom0, single_n, _ = clean_stats(single)
multi_mom0, multi_minor, multi_comps = clean_stats(multi)

rmsf_fwhm = float(synth.fdf_parameters["fwhm_rmsf_radm2"][0])
phi_max = float(synth.fdf_parameters["phi_max_scale_radm2"][0])
grid = auto_grid(synth)
print(f"GMIMS-DRAGONS RMSF FWHM = {rmsf_fwhm:.1f} rad/m^2")
print(
    f"max recoverable scale = {phi_max:.0f} rad/m^2 ({phi_max / rmsf_fwhm:.1f} x FWHM)"
)
print(f"auto scale grid (FWHM units) = {list(grid)}")
print(
    f"resolved case dRM = {DELTA_RESOLVED_GMIMS * rmsf_fwhm:.0f} rad/m^2 "
    f"({DELTA_RESOLVED_GMIMS * rmsf_fwhm / phi_max:.0%} of max scale)"
)

print(f"single: mom0={single_mom0:.3f} iterations={single_n}")
print(
    f"multi:  mom0={multi_mom0:.3f} minor_cycles={multi_minor} "
    f"component_steps={multi_comps}"
)

# The grid is extended here (not degenerate): multiscale has scales to engage.
assert len(grid) > 1
# On the thin source it still matches single-scale (the point stays compact).
assert abs(multi_mom0 - single_mom0) < 0.05
assert 0.40 < multi_mom0 < 0.60

In [ ]:
plot_clean(synth, single, multi, "thin", 0.0, rmsf_fwhm, "GMIMS-DRAGONS: thin")

### Marginally resolved ($\Delta\phi \approx 1\times$ FWHM)

In [ ]:
delta_rm = DELTA_MARGINAL * rmsf_fwhm

pol = (
    burn_slab(lambda_sq, delta_rm)
    + rng.normal(0, RMS_NOISE, freq_hz.size)
    + 1j * rng.normal(0, RMS_NOISE, freq_hz.size)
).astype(np.complex128)
err = np.ones_like(pol) * (RMS_NOISE + 1j * RMS_NOISE)
with quiet_logs(logging.ERROR):
    synth = rmsynth.run_rmsynth(freq_hz, pol, err, n_samples=20, do_fit_rmsf=True)
    single = rmclean.run_rmclean_from_synth(
        synth, auto_mask=AUTO_MASK, auto_threshold=AUTO_THRESHOLD
    )
    multi = rmclean.run_rmclean_from_synth(
        synth,
        auto_mask=AUTO_MASK,
        auto_threshold=AUTO_THRESHOLD,
        multiscale=True,
        multiscale_scale_bias=SCALE_BIAS,
        multiscale_max_iter_sub_minor=MULTISCALE_MAX_ITER_SUB_MINOR,
    )

single_mom0, single_n, _ = clean_stats(single)
multi_mom0, multi_minor, multi_comps = clean_stats(multi)

print(f"dRM={delta_rm:.0f} rad/m^2 ({DELTA_MARGINAL:.0f}x FWHM)")
print(f"single: mom0={single_mom0:.3f} iterations={single_n}")
print(
    f"multi:  mom0={multi_mom0:.3f} minor_cycles={multi_minor} "
    f"component_steps={multi_comps}"
)

# A source only ~1 RMSF FWHM wide is barely resolved, but the bias still throws
# it onto the first extended scale (wider than the source). There the
# non-orthogonal kernels double-count its integrated flux, so multiscale
# over-recovers, landing a few tens of percent above single-scale. This is the
# known multiscale flux-accounting limitation, not a bug (see the summary).
assert single_mom0 < multi_mom0 < 1.5 * single_mom0

In [ ]:
plot_clean(synth, single, multi, "slab", delta_rm, rmsf_fwhm, "GMIMS-DRAGONS: marginal")

### Resolved thick ($\Delta\phi \approx 6\times$ FWHM)

Now the source is genuinely resolved and well inside the recoverable window. This
is where multiscale engages the extended scales: it spreads its model across the
slab (many channels carry model flux) instead of piling deltas at a point, and it
recovers at least as much integrated flux as single-scale, keeping the thick flux
that single-scale would otherwise leave in the residual. The Burn slab still
depolarises, so neither recovers the full intrinsic flux.

In [ ]:
delta_rm = DELTA_RESOLVED_GMIMS * rmsf_fwhm

pol = (
    burn_slab(lambda_sq, delta_rm)
    + rng.normal(0, RMS_NOISE, freq_hz.size)
    + 1j * rng.normal(0, RMS_NOISE, freq_hz.size)
).astype(np.complex128)
err = np.ones_like(pol) * (RMS_NOISE + 1j * RMS_NOISE)
with quiet_logs(logging.ERROR):
    synth = rmsynth.run_rmsynth(freq_hz, pol, err, n_samples=20, do_fit_rmsf=True)
    single = rmclean.run_rmclean_from_synth(
        synth, auto_mask=AUTO_MASK, auto_threshold=AUTO_THRESHOLD
    )
    multi = rmclean.run_rmclean_from_synth(
        synth,
        auto_mask=AUTO_MASK,
        auto_threshold=AUTO_THRESHOLD,
        multiscale=True,
        multiscale_scale_bias=SCALE_BIAS,
        multiscale_max_iter_sub_minor=MULTISCALE_MAX_ITER_SUB_MINOR,
    )

single_mom0, single_n, _ = clean_stats(single)
multi_mom0, multi_minor, multi_comps = clean_stats(multi)
single_span, multi_span = model_span(single), model_span(multi)
print(
    f"dRM={delta_rm:.0f} rad/m^2 ({DELTA_RESOLVED_GMIMS:.0f}x FWHM, "
    f"{delta_rm / phi_max:.0%} of max scale)"
)
print(f"single: mom0={single_mom0:.3f} iterations={single_n} model spans {single_span}")
print(
    f"multi:  mom0={multi_mom0:.3f} minor_cycles={multi_minor} "
    f"component_steps={multi_comps} (model spans {multi_span} channels)"
)

# Multiscale recovers more of the thick flux: its model spreads across the slab
# (far more channels carry model flux) instead of sitting at a point, and its
# integrated flux is at least as high as single-scale.
assert multi_span > single_span
assert multi_mom0 > 0.95 * single_mom0

In [ ]:
plot_clean(
    synth, single, multi, "slab", delta_rm, rmsf_fwhm, "GMIMS-DRAGONS: resolved thick"
)

### Same separation, two thin points

The same $\Delta\phi$ separation as the resolved slab, but two thin components.
These do not depolarise, so both algorithms recover the full flux. Compare the
model (bottom row) with the slab's: the slab draws in broad extended-scale
components, whereas the two points are closer to compact. Multiscale still
spreads and over-recovers a little here (the interference structure between the
two RMSF responses looks partly extended to the scale kernels), but far less
than for the genuine slab, so thickness, not mere separation, is what pulls in
the large scales.


In [ ]:
pol = (
    two_thin_points(lambda_sq, delta_rm)
    + rng.normal(0, RMS_NOISE, freq_hz.size)
    + 1j * rng.normal(0, RMS_NOISE, freq_hz.size)
).astype(np.complex128)
err = np.ones_like(pol) * (RMS_NOISE + 1j * RMS_NOISE)
with quiet_logs(logging.ERROR):
    synth = rmsynth.run_rmsynth(freq_hz, pol, err, n_samples=20, do_fit_rmsf=True)
    single = rmclean.run_rmclean_from_synth(
        synth, auto_mask=AUTO_MASK, auto_threshold=AUTO_THRESHOLD
    )
    multi = rmclean.run_rmclean_from_synth(
        synth,
        auto_mask=AUTO_MASK,
        auto_threshold=AUTO_THRESHOLD,
        multiscale=True,
        multiscale_scale_bias=SCALE_BIAS,
        multiscale_max_iter_sub_minor=MULTISCALE_MAX_ITER_SUB_MINOR,
    )

single_mom0, single_n, _ = clean_stats(single)
multi_mom0, multi_minor, multi_comps = clean_stats(multi)

multi_model = np.abs(multi.fdf_arrs["fdf_model_complex_arr"].to_numpy().astype(complex))
multi_model_chan = int((multi_model > 0.01 * multi_model.max()).sum())
print(f"two thin points separated by {delta_rm:.0f} rad/m^2")
print(f"single: mom0={single_mom0:.3f} iterations={single_n}")
print(
    f"multi:  mom0={multi_mom0:.3f} minor_cycles={multi_minor} "
    f"component_steps={multi_comps} (model spans {multi_model_chan} channels)"
)

# Two points do not depolarise: both recover near-full flux.
assert multi_mom0 > 0.85 * FRAC_POL

In [ ]:
plot_clean(
    synth,
    single,
    multi,
    "points",
    delta_rm,
    rmsf_fwhm,
    "GMIMS-DRAGONS: two thin points (same separation)",
)

## When does multiscale actually help?

Every case above uses a physical Burn slab, which *depolarises*: its $Q,U$ signal
is attenuated at high $\lambda^2$, so a Faraday-thick source's flux is genuinely
reduced in the data. **"Extended in Faraday depth" means "depolarised"** - unlike
synthesis imaging, the missing flux is gone, and no CLEAN variant recovers it.
That is why multiscale matches single-scale on flux throughout.

Multiscale's value is the *model*: once extended structure is detected, it uses a
few broad components instead of a picket fence of deltas. To see this, use a
smooth Gaussian FDF (external Faraday dispersion) through real GMIMS-wide coverage
with noise added in the **channel domain** (per-frequency $Q,U$) via the shared
`rm_lite.utils.simulate` helpers, so the FDF noise is the physically correct
correlated field, not iid noise painted onto $\phi$. The first figure is the
channel spectrum: $|P|$ depolarises with $\lambda^2$ about $\lambda^2 = 0$ and
stays below the intrinsic fraction, with per-channel noise on top.

At matched depth multiscale recovers **comparable flux** (within ~15%, not more)
and spreads its model across **many more channels** as a few broad components, at
the cost of **more** component steps. It does *not* clean deeper: the on-source
residual is comparable to, or slightly above, single-scale (see the printed
values). The value is the source-matched model, not a whiter residual or extra
flux.

One practical note: give multiscale a **generous $\phi$ window**, since scale
kernels wider than the window pick up reflect-mode edge artefacts (`simulate`
uses a 40 FWHM window). The mask confines cleaning to significant channels, so a
deep threshold (here $0.5\sigma$) is safe.

In [ ]:
from rm_lite.utils.clean import rmclean as rmclean_lowlevel
from rm_lite.utils.fitting import unit_centred_gaussian
from rm_lite.utils.simulate import gauss, simulate_fdf
from rm_lite.utils.synthesis import calc_faraday_moments

# External Faraday dispersion (Gaussian FDF) through real GMIMS-wide coverage,
# noise added per channel. amp is the polarised fraction, so |P| is physical.
GAUSS_SIGMA_FWHM = 1.0  # Gaussian FDF sigma in RMSF FWHM units
SN_EXT = 5.0  # S/N on the polarised fraction
gmims_wide = np.linspace(0.3e9, 1.8e9, 500)

spec_ext = gauss(sigma_fwhm=GAUSS_SIGMA_FWHM, amp=FRAC_POL)
sim = simulate_fdf(spec_ext, gmims_wide, rng=np.random.default_rng(0), sn=SN_EXT)
sim_clean = simulate_fdf(spec_ext, gmims_wide)  # noise-free, for the envelope
sigma_rm = GAUSS_SIGMA_FWHM * sim.fwhm
true_ext = FRAC_POL * unit_centred_gaussian(sim.phi_arr_radm2, stddev=sigma_rm)

# Channel spectrum going into CLEAN: |P| depolarises with lambda^2, noise on top.
assert np.abs(sim_clean.complex_pol_arr).max() <= FRAC_POL + 1e-6
figc, axc = plt.subplots(figsize=(7.5, 3.2))
axc.plot(
    sim.lambda_sq_arr_m2,
    sim.complex_pol_arr.real,
    color="tab:blue",
    lw=0.6,
    alpha=0.7,
    label="Q (noisy)",
)
axc.plot(
    sim.lambda_sq_arr_m2,
    sim.complex_pol_arr.imag,
    color="tab:orange",
    lw=0.6,
    alpha=0.7,
    label="U (noisy)",
)
axc.plot(
    sim.lambda_sq_arr_m2,
    np.abs(sim.complex_pol_arr),
    color="0.6",
    lw=0.6,
    alpha=0.6,
    label="|P| (noisy)",
)
axc.plot(
    sim_clean.lambda_sq_arr_m2,
    np.abs(sim_clean.complex_pol_arr),
    color="k",
    lw=1.8,
    label="|P| (noise-free envelope)",
)
axc.set(
    xlabel=rf"$\lambda^2$ / {u.m**2:latex_inline}",
    ylabel="fractional polarisation",
    title=f"Channel spectrum into CLEAN (external dispersion, S/N = {SN_EXT:.0f})",
)
axc.legend(fontsize=7, ncol=2, loc="upper right")
figc.tight_layout()

results_ext = {}
for use_multiscale in (False, True):
    with quiet_logs(logging.ERROR):
        results_ext[use_multiscale] = rmclean_lowlevel(
            dirty_fdf_arr=sim.dirty_fdf,
            phi_arr_radm2=sim.phi_arr_radm2,
            rmsf_arr=sim.rmsf_arr,
            phi_double_arr_radm2=sim.phi_double_arr_radm2,
            fwhm_rmsf_arr=np.array(sim.fwhm),
            mask=7 * sim.fdf_noise,
            threshold=0.5 * sim.fdf_noise,
            max_iter=5000,
            multiscale=use_multiscale,
            multiscale_max_iter_sub_minor=MULTISCALE_MAX_ITER_SUB_MINOR,
        )

on_source = np.abs(sim.phi_arr_radm2) <= 3 * sigma_rm
metrics = {}
for use_multiscale, name in [(False, "single"), (True, "multi")]:
    res = results_ext[use_multiscale]
    clean = np.abs(res.clean_fdf_arr)
    resid = np.abs(res.resid_fdf_arr)
    model = np.abs(res.model_fdf_arr)
    metrics[name] = {
        "mom0": float(calc_faraday_moments(clean, sim.phi_arr_radm2, sim.fwhm).mom0),
        "resid_on_max": float(resid[on_source].max() / sim.fdf_noise),
        "comps": int(np.ravel(res.sub_minor_iter_arr)[0]),
        "model_chan": int((model > 0.01 * model.max()).sum()),
    }
    print(
        f"{name:>6}: mom0={metrics[name]['mom0']:.3f} "
        f"resid on-source max/noise={metrics[name]['resid_on_max']:.2f} "
        f"component_steps={metrics[name]['comps']} "
        f"model spans {metrics[name]['model_chan']} channels"
    )

# Comparable flux, a much broader model, more components - not a lower residual.
assert 0.8 < metrics["multi"]["mom0"] / metrics["single"]["mom0"] < 1.2
assert metrics["multi"]["model_chan"] > 3 * metrics["single"]["model_chan"]
assert metrics["multi"]["comps"] > metrics["single"]["comps"]

# FDF (top, normalised to the dirty peak) and residual real part (bottom).
fig, axs = plt.subplots(2, 2, figsize=(9.5, 6), sharex=True)
half = 6 * sigma_rm
phi = sim.phi_arr_radm2
norm = float(np.abs(sim.dirty_fdf).max())
for col, (use_multiscale, name) in enumerate(
    [(False, "single-scale"), (True, "multiscale")]
):
    res = results_ext[use_multiscale]
    comps = int(np.ravel(res.sub_minor_iter_arr)[0])
    ax = axs[0, col]
    ax.plot(
        phi,
        true_ext / true_ext.max(),
        color="tab:orange",
        ls="--",
        lw=1.3,
        label="true FDF (source, normalised)",
        zorder=1000,
    )
    ax.plot(
        phi,
        np.abs(sim.dirty_fdf) / norm,
        color="0.6",
        lw=1.2,
        label="dirty FDF (before CLEAN)",
    )
    ax.plot(
        phi,
        np.abs(res.clean_fdf_arr) / norm,
        color="k",
        lw=1.4,
        label="cleaned FDF (restored)",
    )
    ax.step(
        phi,
        np.abs(res.model_fdf_arr) / norm,
        color="tab:red",
        where="mid",
        lw=0.9,
        label="CLEAN model components",
    )
    ax.set(ylabel="|FDF| (norm. to dirty peak)")
    ax.text(
        0.02,
        0.96,
        f"{name}: {comps} component steps",
        transform=ax.transAxes,
        va="top",
        fontweight="bold",
    )
    axr = axs[1, col]
    axr.axhspan(
        -0.5 * sim.fdf_noise,
        0.5 * sim.fdf_noise,
        color="tab:green",
        alpha=0.15,
        label=r"$\pm 0.5\sigma$ clean threshold",
    )
    axr.axhline(0.0, color="0.7", lw=0.8, label="zero (ideal noise-like residual)")
    axr.plot(
        phi,
        res.resid_fdf_arr.real,
        color="tab:blue",
        lw=0.7,
        label="residual FDF (real part)",
    )
    axr.set(
        xlabel=rf"$\phi$ / {u.rad / u.m**2:latex_inline}",
        ylabel="residual (real part)",
        xlim=(-half, half),
    )
axs[0, 0].legend(fontsize=7, loc="upper right")
axs[1, 0].legend(fontsize=7, loc="upper right")
fig.suptitle(
    "Detected extended FDF (Gaussian), real GMIMS-wide RMSF: single-scale vs multiscale"
)
fig.tight_layout()

## Uniform-$\lambda^2$ coverage

Real receivers sample uniformly in frequency, which clusters the $\lambda^2$
samples and gives the RMSF strong sidelobes. Sampling uniformly in $\lambda^2$ is
contrived but gives the cleanest RMSF, a useful reference. On a thick slab both
algorithms recover comparable flux (the slab still depolarises); multiscale uses
fewer minor cycles but, as everywhere, not fewer component steps. We will revisit
this once low-$\lambda^2$ up-weighting (the `new-weights` schemes) lands:
weighting the near-DC channels up should lift the extended Faraday structure out
of the noise and let multiscale's model advantage show more clearly here.</cell id="91658be0">

In [ ]:
from rm_lite.utils.synthesis import lambda2_to_freq

# Uniform in lambda^2 (contrived) -> the cleanest RMSF. A generous phi window
# (phi_max_radm2) gives the extended model room; narrow windows breed edge
# artefacts in multiscale.
lambda_sq_uniform = np.linspace(0.001, 1.0, 400)
freq_uniform = lambda2_to_freq(lambda_sq_uniform)

with quiet_logs(logging.ERROR):
    synth_u_thin = rmsynth.run_rmsynth(
        freq_uniform,
        burn_slab(lambda_sq_uniform, 0.0),
        np.ones_like(lambda_sq_uniform) * (RMS_NOISE + 1j * RMS_NOISE),
        phi_max_radm2=600.0,
        do_fit_rmsf=True,
    )
rmsf_fwhm_u = float(synth_u_thin.fdf_parameters["fwhm_rmsf_radm2"][0])
delta_rm_u = 2 * rmsf_fwhm_u

pol = (
    burn_slab(lambda_sq_uniform, delta_rm_u)
    + rng.normal(0, RMS_NOISE, freq_uniform.size)
    + 1j * rng.normal(0, RMS_NOISE, freq_uniform.size)
).astype(np.complex128)
err = np.ones_like(pol) * (RMS_NOISE + 1j * RMS_NOISE)
with quiet_logs(logging.ERROR):
    synth = rmsynth.run_rmsynth(
        freq_uniform, pol, err, phi_max_radm2=600.0, do_fit_rmsf=True
    )
    single = rmclean.run_rmclean_from_synth(
        synth, auto_mask=AUTO_MASK, auto_threshold=3
    )
    multi = rmclean.run_rmclean_from_synth(
        synth,
        auto_mask=AUTO_MASK,
        auto_threshold=3,
        multiscale=True,
        multiscale_scale_bias=SCALE_BIAS,
        multiscale_max_iter_sub_minor=MULTISCALE_MAX_ITER_SUB_MINOR,
    )

single_mom0, single_n, _ = clean_stats(single)
multi_mom0, multi_minor, multi_comps = clean_stats(multi)
print(
    f"uniform-lambda^2 RMSF FWHM = {rmsf_fwhm_u:.1f} rad/m^2, "
    f"dRM = {delta_rm_u:.0f} rad/m^2"
)
print(f"single: mom0={single_mom0:.3f} iterations={single_n}")
print(
    f"multi:  mom0={multi_mom0:.3f} minor_cycles={multi_minor} "
    f"component_steps={multi_comps}"
)

# Comparable flux (the slab still depolarises).
assert multi_mom0 > 0.75 * single_mom0

plot_clean(
    synth,
    single,
    multi,
    "slab",
    delta_rm_u,
    rmsf_fwhm_u,
    r"Uniform-$\lambda^2$: thick slab",
)

## Regime C: forcing scales on narrow coverage

If you know your field has extended Faraday structure and want multiscale on a
narrow band anyway, pass explicit `multiscale_scales` (RMSF FWHM units). This
overrides the auto grid and its degeneration guard.

Be honest about what this buys. On narrow coverage the extended structure is
depolarised out of the data (Regime A), so forcing scales does not recover flux
that is not there: it changes the *model* (broad components instead of a delta)
while conserving total flux. Use it deliberately, not as a default.

In [ ]:
# Narrow coverage again (RACS-low, degenerate auto grid), now with an explicit
# extended grid forced on.
freq_c = np.linspace(744e6, 1032e6, 144)
lsq_c = freq_to_lambda2(freq_c)
pol = (
    burn_slab(lsq_c, 60.0)  # a slab ~1 RMSF FWHM wide
    + rng.normal(0, RMS_NOISE, freq_c.size)
    + 1j * rng.normal(0, RMS_NOISE, freq_c.size)
).astype(np.complex128)
err = np.ones_like(pol) * (RMS_NOISE + 1j * RMS_NOISE)
with quiet_logs(logging.ERROR):
    synth = rmsynth.run_rmsynth(freq_c, pol, err, n_samples=20, do_fit_rmsf=True)
    auto = rmclean.run_rmclean_from_synth(
        synth,
        auto_mask=AUTO_MASK,
        auto_threshold=AUTO_THRESHOLD,
        multiscale=True,
        multiscale_max_iter_sub_minor=MULTISCALE_MAX_ITER_SUB_MINOR,
    )
    forced = rmclean.run_rmclean_from_synth(
        synth,
        auto_mask=AUTO_MASK,
        auto_threshold=AUTO_THRESHOLD,
        multiscale=True,
        multiscale_scales=np.array([0.0, 2.0, 4.0]),
        multiscale_max_iter_sub_minor=MULTISCALE_MAX_ITER_SUB_MINOR,
    )

auto_mom0 = float(auto.fdf_parameters["mom0_debias"][0])
forced_mom0 = float(forced.fdf_parameters["mom0_debias"][0])
print(f"auto (degenerate [0.0]): mom0={auto_mom0:.3f} model spans {model_span(auto)}")
print(
    f"forced [0.0, 2.0, 4.0]:  mom0={forced_mom0:.3f} model spans {model_span(forced)}"
)

# Forcing the grid engages extended kernels (a broader model) ...
assert model_span(forced) > model_span(auto)
# ... but conserves flux: it does not invent the depolarised structure back.
assert np.isclose(forced_mom0, auto_mom0, atol=0.05)

## Summary

Multiscale RM-CLEAN is a wideband tool for Faraday-*thick* structure. Whether it
does anything is set by the coverage, through the auto scale grid:

| Regime | Coverage | Auto grid | Multiscale behaviour |
|---|---|---|---|
| A: degenerate | single narrow band (RACS-low) | `[0.0]` | reduces to single-scale (warned) |
| B: engaging | wide, contiguous (GMIMS 0.3-1.8 GHz) | `[0.0, 4.0, 8.0]` | models thick flux as extended components |
| C: forced | narrow band + explicit `multiscale_scales` | user-set | broader model, flux conserved, no new flux |

The key points:

- **The grid is the lever.** When $\phi_{\rm max} = \pi / \lambda^2_{\rm min}$
  leaves no room for a scale kernel, the grid collapses to `[0.0]` and multiscale
  is single-scale. `default_scales` warns when this happens.
- **"Extended in Faraday depth" means "depolarised".** A Burn slab's $Q,U$ signal
  shrinks at high $\lambda^2$, so a thick source's flux is genuinely reduced in
  the data. No CLEAN variant recovers what is gone; on wide coverage multiscale
  recovers what *survives* and models it as a few broad components rather than a
  picket fence.
- **Flux is honest but not exact.** The scale kernels are non-orthogonal, so a
  source thrown onto a scale wider than itself has its integrated flux
  double-counted; a marginally-resolved source can over-recover by tens of
  percent. On a wide coverage the fixed bias can over-extend a compact source
  too: a known limitation, not fixed here.
- **Not fewer components.** Multiscale converges in fewer *minor cycles* but
  places at least as many component-placement steps as single-scale.
- **The genuine value is the model:** source-matched (extended for thick
  structure, compact for a point) rather than a picket fence of deltas.

Enable multiscale with `multiscale=True` on `run_rmclean_from_synth` (1D) or
`rmclean_3d_from_synth` (3D). Give it wide, contiguous coverage or an explicit
`multiscale_scales`, and a generous $\phi$ window (scale kernels wider than the
window pick up edge artefacts); the mask confines cleaning to significant
channels, so a deep threshold (these runs use $0.5\sigma$) is safe. Tune with
`multiscale_scale_bias` (default 0.6; towards 1 reduces to single-scale, below it
over-extends), `multiscale_scales`, `multiscale_max_iter_sub_minor`, and the usual
`auto_mask` / `auto_threshold`.